In [ ]:
import math
import shutil
import subprocess
from pathlib import Path

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds
import seamless_3dep as s3dep
import whitebox

nlcd_master_path = "/kaggle/input/datasets/maanav0114/ambystoma-sdm-data/land_cover_clipped_reproj.tiff"
MASTER_NODATA = 255      # NLCD layers are uint8 with 255 as nodata
NLCD_OPEN_WATER = 11     # NLCD class 11 -> no meaningful slope, excluded from TWI
DEM_NODATA = -9999.0     # continuous layers use -9999, NLCD layers use 255

work_dir = Path("/kaggle/working/terrain_data")
tile_dir = work_dir / "dem_tiles"
work_dir.mkdir(parents=True, exist_ok=True)
tile_dir.mkdir(parents=True, exist_ok=True)


def free_gb(path):
    """Free space on the filesystem holding path, in GB."""
    return shutil.disk_usage(path).free / 1e9


print(f"Working dir : {work_dir} ({free_gb(work_dir):.1f} GB free)")

vrt_file = str(work_dir / "merged_4326.vrt")
dem_work = str(work_dir / "six_state_dem_twi_grid.tif")
dem_breached = str(work_dir / "six_state_dem_breached.tif")
slope_deg = str(work_dir / "six_state_slope_deg.tif")
slope_floored = str(work_dir / "six_state_slope_deg_floored.tif")
sca_file = str(work_dir / "six_state_sca.tif")
twi_work = str(work_dir / "six_state_twi_working_grid.tif")
final_twi = str(work_dir / "six_state_twi_aligned.tif")

TWI_RES = 120.0
MAX_WORK_CELLS = 400_000_000

# WhiteboxTools writes a one-cell nodata frame around every raster it produces.
# Padding the working grid by this many cells keeps that frame outside the
# master extent, so no valid 30 m pixel is lost to the tool's own edge.
WORK_PAD_CELLS = 2

BREACH_DIST_M = 60.0
BREACH_DIST_CELLS = max(1, int(round(BREACH_DIST_M / TWI_RES)))

MIN_TAN_BETA = 0.001
MIN_SLOPE_DEG = math.degrees(math.atan(MIN_TAN_BETA))

with rasterio.open(nlcd_master_path) as master:
    master_crs = master.crs.to_string()
    minx, miny, maxx, maxy = master.bounds
    width = master.width
    height = master.height
    master_transform = master.transform

print(f"Master grid : {width} x {height} px @ {abs(master_transform.a):.0f} m | "
      f"CRS {master_crs} | nodata {MASTER_NODATA}")

work_pad = WORK_PAD_CELLS * TWI_RES
work_minx = math.floor(minx / TWI_RES) * TWI_RES - work_pad
work_miny = math.floor(miny / TWI_RES) * TWI_RES - work_pad
work_maxx = math.ceil(maxx / TWI_RES) * TWI_RES + work_pad
work_maxy = math.ceil(maxy / TWI_RES) * TWI_RES + work_pad
work_width = int(round((work_maxx - work_minx) / TWI_RES))
work_height = int(round((work_maxy - work_miny) / TWI_RES))
work_cells = work_width * work_height

print(f"Working grid: {work_width} x {work_height} px @ {TWI_RES:.0f} m = "
      f"{work_cells/1e6:.1f} M cells (~{work_cells*4/1e9:.2f} GB per float32 layer)")
print(f"              padded {WORK_PAD_CELLS} cell(s) = {work_pad:.0f} m beyond the master extent")
if work_cells > MAX_WORK_CELLS:
    raise ValueError(
        f"Working grid has {work_cells/1e9:.2f}e9 cells, above the "
        f"{MAX_WORK_CELLS/1e9:.2f}e9 budget. Raise TWI_RES or tile the DEM."
    )

pad = 1000.0
if pad < work_pad:
    raise ValueError(
        f"DEM download pad {pad:.0f} m is smaller than the {work_pad:.0f} m "
        f"working-grid pad; the padded edges would be left without DEM coverage."
    )
bbox_4326 = transform_bounds(
    master_crs, "EPSG:4326",
    minx - pad, miny - pad, maxx + pad, maxy + pad
)


In [ ]:
print("Downloading 30 m DEM tiles from USGS 3DEP...")
tiff_files = s3dep.get_dem(bbox_4326, tile_dir, res=30, buff_npixels=2)
print(f"{len(tiff_files)} 3DEP tile(s) in {tile_dir}")

s3dep.build_vrt(vrt_file, tiff_files)
assert Path(vrt_file).exists(), f"3DEP VRT was not created: {vrt_file}"
print(f"Mosaic: {vrt_file}")


In [ ]:
print(f"Warping DEM onto the {TWI_RES:.0f} m working grid...")
subprocess.run([
    "gdalwarp",
    "-t_srs", master_crs,
    "-te", str(work_minx), str(work_miny), str(work_maxx), str(work_maxy),
    "-tr", str(TWI_RES), str(TWI_RES),
    "-r", "bilinear",
    "-dstnodata", str(DEM_NODATA),
    "-ot", "Float32",
    "-co", "COMPRESS=LZW",
    "-co", "BIGTIFF=YES",
    "-co", "TILED=YES",
    "-multi", "-wo", "NUM_THREADS=ALL_CPUS",
    "-overwrite",
    vrt_file,
    dem_work,
], check=True)

with rasterio.open(dem_work) as src:
    if (src.width, src.height) != (work_width, work_height):
        raise RuntimeError(
            f"DEM grid {src.width} x {src.height} != expected {work_width} x {work_height}"
        )
    print(f"  -> {src.width} x {src.height} | CRS {src.crs} | "
          f"nodata {src.nodata} | {src.dtypes[0]}")

# A six-state 30 m mosaic is many GB and nothing downstream reads it again.
dropped_bytes = sum(f.stat().st_size for f in tile_dir.rglob("*") if f.is_file())
dropped_bytes += Path(vrt_file).stat().st_size
shutil.rmtree(tile_dir)
tile_dir.mkdir(parents=True, exist_ok=True)
Path(vrt_file).unlink(missing_ok=True)
print(f"  freed {dropped_bytes/1e9:.1f} GB of 30 m tiles + VRT | "
      f"{free_gb(work_dir):.1f} GB free")


In [ ]:
wbt = whitebox.WhiteboxTools()
wbt.set_working_dir(str(work_dir))
wbt.set_verbose_mode(False)


def run_wbt(label, out_path, tool, **kwargs):
    """Run one WhiteboxTools tool and fail loudly if it did not produce output.

    WhiteboxTools.run_tool() *returns* 1 on error instead of raising, which is how
    the previous version printed "Done!" after every step had already crashed.
    """
    Path(out_path).unlink(missing_ok=True)
    rc = tool(**kwargs)
    if rc != 0:
        raise RuntimeError(f"{label} failed (WhiteboxTools exit code {rc})")
    if not Path(out_path).exists() or Path(out_path).stat().st_size == 0:
        raise RuntimeError(f"{label} returned success but did not write {out_path}")
    print(f"  ok  {label} -> {Path(out_path).name} "
          f"({Path(out_path).stat().st_size/1e6:.1f} MB)")


def map_band(src_path, dst_path, func):
    """Apply func(band, nodata) to one band, block by block, keeping the grid."""
    with rasterio.open(src_path) as src:
        nodata = src.nodata
        profile = src.profile.copy()
        profile.update({
            "count": 1,
            "dtype": "float32",
            "nodata": nodata,
            "compress": "LZW",
            "tiled": True,
            "blockxsize": 512,
            "blockysize": 512,
            "BIGTIFF": "YES",
        })
        with rasterio.open(dst_path, "w", **profile) as dst:
            for _, window in dst.block_windows(1):
                data = src.read(1, window=window)
                dst.write(func(data, nodata).astype("float32"), 1, window=window)


def floor_slope(data, nodata):
    """Raise every valid slope to at least MIN_SLOPE_DEG, leaving nodata alone."""
    out = data.astype("float32", copy=True)
    keep = np.isfinite(out)
    if nodata is not None:
        keep &= out != nodata
    out[keep] = np.maximum(out[keep], MIN_SLOPE_DEG)
    return out


print("Conditioning DEM and calculating TWI...")

# The DEM is deliberately unmasked here. Blanking open water on the 120 m grid
# punched 120 m holes along every stream and shoreline, which severed upslope
# flow routing; the 30 m study-area/water mask is applied after alignment instead.
run_wbt(
    f"BreachDepressionsLeastCost (dist={BREACH_DIST_CELLS} cells = "
    f"{BREACH_DIST_CELLS * TWI_RES:.0f} m)",
    dem_breached,
    wbt.breach_depressions_least_cost,
    dem=dem_work,
    output=dem_breached,
    dist=BREACH_DIST_CELLS,
    min_dist=True,
    fill=True,
)

run_wbt(
    "Slope",
    slope_deg,
    wbt.slope,
    dem=dem_breached,
    output=slope_deg,
    units="degrees",
)

map_band(slope_deg, slope_floored, floor_slope)
print(f"  ok  slope floored at tan(beta) >= {MIN_TAN_BETA} "
      f"({MIN_SLOPE_DEG:.4f} deg) -> {Path(slope_floored).name}")

run_wbt(
    "DInfFlowAccumulation",
    sca_file,
    wbt.d_inf_flow_accumulation,
    i=dem_breached,
    output=sca_file,
    out_type="Specific Contributing Area",
)

run_wbt(
    "WetnessIndex",
    twi_work,
    wbt.wetness_index,
    sca=sca_file,
    slope=slope_floored,
    output=twi_work,
)


In [ ]:
with rasterio.open(twi_work) as src:
    twi_src_nodata = src.nodata
twi_src_nodata_arg = str(twi_src_nodata if twi_src_nodata is not None else DEM_NODATA)
print(f"WhiteboxTools TWI nodata: {twi_src_nodata} -> resampling onto the master grid...")
print(f"Aligned grid: {width} x {height} px (~{width * height * 4 / 1e9:.1f} GB as Float32) | "
      f"{free_gb(work_dir):.1f} GB free")

subprocess.run([
    "gdalwarp",
    "-t_srs", master_crs,
    "-te", str(minx), str(miny), str(maxx), str(maxy),
    "-ts", str(width), str(height),
    "-r", "bilinear",
    "-srcnodata", twi_src_nodata_arg,
    "-dstnodata", str(DEM_NODATA),
    "-ot", "Float32",
    "-co", "COMPRESS=LZW",
    "-co", "PREDICTOR=3",      # float predictor: much smaller than plain LZW
    "-co", "BIGTIFF=YES",
    "-co", "TILED=YES",
    "-co", "BLOCKXSIZE=512",
    "-co", "BLOCKYSIZE=512",
    "-multi", "-wo", "NUM_THREADS=ALL_CPUS",
    "-overwrite",
    twi_work,
    final_twi,
], check=True)

# The 120 m terrain stack is finished once TWI is on the master grid; dropping it
# here frees the space the 30 m alignment needs.
freed = 0
for stale in (twi_work, sca_file, slope_floored, dem_breached, slope_deg, dem_work):
    stale_path = Path(stale)
    if stale_path.exists():
        freed += stale_path.stat().st_size
        stale_path.unlink()
print(f"  dropped {freed/1e9:.2f} GB of 120 m intermediates | "
      f"{free_gb(work_dir):.1f} GB free")

# Share of study-area land allowed to come out without a TWI value before the run
# complains. 3DEP voids and the WBT edge frame make some loss legitimate.
MISSING_WARN_FRAC = 0.01

kept = dropped_water = dropped_outside = 0
missing_by_class = np.zeros(256, dtype=np.int64)

with rasterio.open(final_twi, "r+") as dst, rasterio.open(nlcd_master_path) as master:
    if (dst.width, dst.height) != (master.width, master.height):
        raise RuntimeError(f"TWI grid {dst.shape} does not match master {master.shape}")
    if not dst.transform.almost_equals(master.transform, precision=1e-6):
        raise RuntimeError("TWI transform drifted from the master grid")
    if dst.crs != master.crs:
        raise RuntimeError(f"TWI CRS {dst.crs} != master CRS {master.crs}")

    # Mask in place: the aligned grid is ~2e9 px (~8 GB as Float32), so writing a
    # second full-size copy is what ran the disk out of space.
    for _, window in dst.block_windows(1):
        twi = dst.read(1, window=window)
        land_cover = master.read(1, window=window)
        # NaN != nodata is True in NumPy, so NaNs would otherwise count as data.
        had_data = np.isfinite(twi) & (twi != DEM_NODATA)
        is_land = (land_cover != MASTER_NODATA) & (land_cover != NLCD_OPEN_WATER)
        good = is_land & had_data

        kept += int(good.sum())
        dropped_water += int((had_data & (land_cover == NLCD_OPEN_WATER)).sum())
        dropped_outside += int((had_data & (land_cover == MASTER_NODATA)).sum())

        # Land inside the study area that has no TWI at all is lost terrain, not
        # a mask effect; count it by NLCD class so holes can be traced.
        missing = is_land & ~had_data
        if missing.any():
            missing_by_class += np.bincount(
                land_cover[missing].astype(np.intp), minlength=256
            )

        twi[~good] = DEM_NODATA
        dst.write(twi, 1, window=window)

missing_total = int(missing_by_class.sum())
land_total = kept + missing_total
print(f"  valid: {kept/1e6:.1f} M px | masked as open water: {dropped_water/1e6:.1f} M px | "
      f"outside the cutline: {dropped_outside/1e6:.1f} M px")

if missing_total:
    top_classes = sorted(
        ((cls, int(n)) for cls, n in enumerate(missing_by_class) if n),
        key=lambda cls_n: cls_n[1],
        reverse=True,
    )[:5]
    missing_frac = missing_total / max(land_total, 1)
    print(f"  study-area land with no TWI: {missing_total/1e6:.2f} M px ({missing_frac:.2%}) | "
          f"top NLCD " + ", ".join(f"{cls}={n/1e6:.2f} M" for cls, n in top_classes))
    if missing_frac > MISSING_WARN_FRAC:
        print(f"WARNING: {missing_frac:.2%} of study-area land came out without a TWI value "
              f"(threshold {MISSING_WARN_FRAC:.0%}); check 3DEP voids and the WBT edge frame")

print(f"Final aligned TWI: {final_twi}")


In [ ]:
import matplotlib.pyplot as plt

with rasterio.open(final_twi) as src, rasterio.open(nlcd_master_path) as master:
    res_m = abs(src.transform.a)
    print(f"{Path(final_twi).name}: {src.width} x {src.height} px @ {res_m:.0f} m | "
          f"CRS {src.crs} | nodata {src.nodata} | {src.dtypes[0]}")

    check_w = min(1500, src.width)
    check_h = min(1500, src.height)
    twi_chk = src.read(1, out_shape=(check_h, check_w), resampling=Resampling.nearest)
    lc_chk = master.read(1, out_shape=(check_h, check_w), resampling=Resampling.nearest)

twi_valid = np.isfinite(twi_chk) & (twi_chk != DEM_NODATA)
master_valid = (lc_chk != MASTER_NODATA) & (lc_chk != NLCD_OPEN_WATER)
leaked = int((twi_valid & ~master_valid).sum())
print(f"mask check  : {leaked} of {int(twi_valid.sum()):,} sampled valid TWI pixels "
      f"fall outside the study area")
if leaked:
    raise RuntimeError("TWI has valid values outside the study-area mask")

with rasterio.open(final_twi) as src:
    scale = min(1500 / src.width, 1500 / src.height)
    out_w = max(1, int(src.width * scale))
    out_h = max(1, int(src.height * scale))
    image = src.read(
        1, out_shape=(out_h, out_w), resampling=Resampling.average, masked=True
    )

values = image.filled(np.nan).astype("float64")
finite = values[np.isfinite(values)]
if finite.size == 0:
    raise RuntimeError("TWI raster has no valid pixels")

lo, p2, p50, p98, hi = np.percentile(finite, [0, 2, 50, 98, 100])
print("TWI         : min {:.2f} | p2 {:.2f} | median {:.2f} | p98 {:.2f} | "
      "max {:.2f}".format(lo, p2, p50, p98, hi))
print(f"             : {np.isfinite(values).mean():.1%} of sampled pixels valid | "
      f"{(finite >= hi - 0.01 * (hi - lo)).mean():.2%} within 1% of the maximum")
if not np.isfinite(finite).all() or hi <= lo:
    raise RuntimeError("TWI distribution is degenerate")
if not 0.0 < p50 < 25.0:
    print("WARNING: median TWI is outside the usual 0-25 range for ln(SCA / tan(beta))")

fig, (ax_map, ax_hist) = plt.subplots(
    1, 2, figsize=(17, 8), gridspec_kw={"width_ratios": [1.35, 1]}
)

im = ax_map.imshow(image, cmap="viridis", vmin=p2, vmax=p98)
ax_map.set_title(f"Topographic Wetness Index  ln(SCA / tan beta)  |  {res_m:.0f} m")
ax_map.axis("off")
fig.colorbar(im, ax=ax_map, fraction=0.036, pad=0.02, label="TWI (stretch: p2-p98)")

ax_hist.hist(finite, bins=150, color="#3f6fa8")
ax_hist.axvline(p50, color="k", ls="--", lw=1)
ax_hist.set_title(f"TWI distribution ({finite.size:,} sampled pixels)")
ax_hist.set_xlabel("TWI")
ax_hist.set_ylabel("pixels")
ax_hist.grid(alpha=0.25)

plt.tight_layout()
plt.show()
